In [21]:
import pandas as pd

df = pd.read_csv(r'C:\Users\moham\OneDrive\Desktop\WebScraping\legacy_app\output\all\all_products_coarsetag.csv')
df['Canonical Text'] = df['Product'] + ' ' + df['Description']
category_dfs = {}

cats = df['coarse_category'].unique()   # ['baby_kids', 'storage', 'bathroom', 'bedding', 'dining', 'clothing', 'other', 'accessories', 'loungeware', 'kitchen', 'home_decor']

for cat in cats:
    # Store each filtered dataframe in the dictionary
    category_dfs[cat] = df[df['coarse_category'] == cat]

In [22]:
# df_cat = category_dfs[cats[0]] # Temporary until code is robust where it will loop through each category

from sentence_transformers import SentenceTransformer, util
import numpy as np
import pandas as pd

# Embedding
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 754.80it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [23]:
for cat in cats:
    df_cat = category_dfs[cat]
    text_descriptions = list(df_cat["Canonical Text"])
    product_embeddings = model.encode(text_descriptions, normalize_embeddings=True)

    # 3. Compute the Matrix
    # This compares the 'product_embeddings' against themselves
    matrix = util.cos_sim(product_embeddings, product_embeddings)
    # Convert to a readable numpy array
    sim_matrix = matrix.numpy()
    sim_matrix
    
    products_similar = [] # Initialize outside the loop
    k = 5

    for i in range(len(sim_matrix)):
        # 1. Sort the entire row: np.argsort returns indices from smallest to largest value
        # 2. [-(k+1):] takes the last k+1 indices (the highest scores)
        # 3. [:-1] excludes the very last one (which is the item itself, score 1.0)
        # 4. [::-1] reverses it so the most similar is first
        indices = np.argsort(sim_matrix[i])[-(k+1):-1][::-1]
        
        # Create the dictionary for this specific row
        dic = {int(idx): df_cat.iloc[idx]["Product"] for idx in indices}
        
        products_similar.append(dic)
    import networkx as nx
    import numpy as np

    cluster_list = []
    # 2. Set your "Agile Threshold"
    threshold = 0.7

    # 1. Create a Graph
    G = nx.Graph()
    G.add_nodes_from(range(len(df_cat)))

    # 2. Add edges based on your Top-K list

    for i, neighbors_dict in enumerate(products_similar):
        # i is the index of the current product
        # neighbors_dict contains the top k indices for that product
        
        for neighbor_idx in neighbors_dict.keys():
            # Check the actual similarity score from the matrix
            score = sim_matrix[i][neighbor_idx]
            
            if score >= threshold:
                # Connect the product to its top-k neighbor
                G.add_edge(i, neighbor_idx)

    # 3. Find the clusters (Connected Components)
    clusters = list(nx.connected_components(G))

    # 4. Map back to Product Names using df_cat
    for cluster in clusters:
        # This maps the local matrix index (0,1,2...) back to the 
        # original CSV row index (e.g. 145, 146...)
        original_indices = [df_cat.index[idx] for idx in cluster]
        cluster_list.append(original_indices)

    print(f"Total Clusters found: {len(cluster_list)}")

    import pandas as pd

    # Initialize the column
    df['Cluster #'] = -1

    for i, c in enumerate(cluster_list):
        # Skip single-item "clusters" if you only want groups of 2+
        if len(c) < 2:
            continue
            
        # 'c' is a list of index labels (e.g., [4, 10, 15])
        # i+1 is the cluster number
        df.loc[c, 'Cluster #'] = i + 1
    filtered_df = df[df['Cluster #'] != -1]
    print(f"Products successfully grouped: {len(filtered_df)}")
    category_dfs[cat] = filtered_df

Total Clusters found: 13
Products successfully grouped: 69
Total Clusters found: 16
Products successfully grouped: 74
Total Clusters found: 8
Products successfully grouped: 49
Total Clusters found: 11
Products successfully grouped: 137
Total Clusters found: 15
Products successfully grouped: 220
Total Clusters found: 50
Products successfully grouped: 291
Total Clusters found: 9
Products successfully grouped: 55
Total Clusters found: 21
Products successfully grouped: 94
Total Clusters found: 1
Products successfully grouped: 2
Total Clusters found: 3
Products successfully grouped: 6


In [24]:
import openpyxl

output_file = r'C:\Users\moham\OneDrive\Desktop\WebScraping\legacy_app\output\all\all_products_divided.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    for cat_name, data in category_dfs.items():
        # Excel sheet names have a 31-character limit
        sheet_name = str(cat_name)[:31]
        
        # Write each dataframe to a specific sheet
        # index=False prevents the row numbers from being saved as a column
        data.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"File saved successfully: {output_file}")



File saved successfully: C:\Users\moham\OneDrive\Desktop\WebScraping\legacy_app\output\all\all_products_divided.xlsx
